In [58]:
function GMRES_IR(A, b; ul = Float16, u = Float32, ur = Float64, up = Float16, ug = Float32, imax=10, atol=1e-6)\
    
    # Step 1: factorization in ul precision
    LU = lu(ul.(A))

    # Step 2: solve x in ul precision
    x = LU \ ul.(b) # should I keep x in higher precision after the operation? 

    for i in 1:imax
        # Step 3: residual in ur precision
        r = ur.(b) - ur.(A) * ur.(x)

        # Step 4: correction in ug precision
        P = (up.(LU.U) \ (up.(LU.L) \ I)) * up.(A)
        p = (ug.(LU.U) \ (ug.(LU.L) \ I)) * ug.(r)

        d, _ = gmres(P, p)
        println(d)

        # Step 5: update in u precision
        x = u.(x) + u.(d)

    end

    return x
end

GMRES_IR (generic function with 1 method)

In [60]:
function example(n, m; ul = Float16, u = Float32, ur = Float64, imax=10, atol=1e-6)

    A = rand(n, m)
    b = rand(n)

    start_time = time()
    x_true = A \ b
    elapsed = time() - start_time
    x_true = vec(x_true)
    println("Run time: ", elapsed)

    start_time = time()
    x = GMRES_IR(A, b)
    elapsed = time() - start_time
    println("Run time: ", elapsed)
    
    error = abs(norm(x-x_true))
    println("Error: ", error)
    

end    

example (generic function with 1 method)

In [62]:
using LinearAlgebra
using BenchmarkTools
using IterativeSolvers

example(100, 100)

Run time: 0.04897189140319824
0.007842871


LoadError: MethodError: no method matching +(::Vector{Float32}, ::Float32)
For element-wise addition, use broadcasting with dot syntax: array .+ scalar
The function `+` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  +(::Any, ::Any, [91m::Any[39m, [91m::Any...[39m)
[0m[90m   @[39m [90mBase[39m [90m[4moperators.jl:596[24m[39m
[0m  +([91m::Bool[39m, ::T) where T<:AbstractFloat
[0m[90m   @[39m [90mBase[39m [90m[4mbool.jl:173[24m[39m
[0m  +([91m::Missing[39m, ::Number)
[0m[90m   @[39m [90mBase[39m [90m[4mmissing.jl:123[24m[39m
[0m  ...
